In [2]:
# Initiate session.
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [8]:
credentials_location = '/home/vytautas.peace/data-engineering-zoomcamp/.google/credentials/spark_gcp.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "/home/vytautas.peace/data-engineering-zoomcamp/06-batch/lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [11]:
# Stop the existing context if it exists
try:
    sc.stop()
except:
    pass

# Now initialize your new one with your GCS config
sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

In [12]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [ ]:
# Run this right after creating your 'spark' session
spark.sparkContext.setLogLevel("ERROR")

In [14]:
df_green = spark.read.parquet('gs://nyc-tlc-data-lake/pq/green/*/*')

df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

df_green.createOrReplaceTempView('green')

In [15]:
# Define Spark query.

df_green_revenue = spark.sql("""
select
    -- Grouping dimensions
    date_trunc('hour', pickup_datetime) as hour,
    PULocationID as zone_id,
    
    sum(total_amount) as green_amount,
    count(1) as green_number_records

from
    green
where
    pickup_datetime >= '2019-01-01 00:00:00'
group by
    1, 2
order by
    1, 2;
""")

In [16]:
df_green_revenue.show(5)

+-------------------+-------+------------------+--------------------+
|               hour|zone_id|      green_amount|green_number_records|
+-------------------+-------+------------------+--------------------+
|2019-01-01 00:00:00|      3|             90.02|                   2|
|2019-01-01 00:00:00|      7| 847.5499999999994|                  68|
|2019-01-01 00:00:00|     11|              10.3|                   1|
|2019-01-01 00:00:00|     14| 77.39999999999999|                   3|
|2019-01-01 00:00:00|     16|50.599999999999994|                   2|
+-------------------+-------+------------------+--------------------+
only showing top 5 rows


In [17]:
df_green_revenue.coalesce(1).write.parquet('gs://nyc-tlc-data-lake/report/revenue/green', mode='overwrite')

In [18]:
df_yellow = spark.read.parquet('gs://nyc-tlc-data-lake/pq/yellow/*/*')

df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

df_yellow.createOrReplaceTempView('yellow')

In [19]:
# Define Spark query.

df_yellow_revenue = spark.sql("""
select
    -- Grouping dimensions
    date_trunc('hour', pickup_datetime) as hour,
    PULocationID as zone_id,
    
    sum(total_amount) as yellow_amount,
    count(1) as yellow_number_records

from
    yellow
where
    pickup_datetime >= '2019-01-01 00:00:00'
group by
    1, 2
order by
    1, 2;
""")

In [20]:
df_yellow_revenue.show(5)

[Stage 17:>                                                         (0 + 4) / 4]

+-------------------+-------+------------------+---------------------+
|               hour|zone_id|     yellow_amount|yellow_number_records|
+-------------------+-------+------------------+---------------------+
|2019-01-01 00:00:00|      4| 731.7199999999998|                   49|
|2019-01-01 00:00:00|      7|332.61000000000007|                   25|
|2019-01-01 00:00:00|     10|              28.6|                    2|
|2019-01-01 00:00:00|     11|              18.6|                    2|
|2019-01-01 00:00:00|     12|             55.55|                    4|
+-------------------+-------+------------------+---------------------+
only showing top 5 rows


In [21]:
df_yellow_revenue.coalesce(1).write.parquet('gs://nyc-tlc-data-lake/report/revenue/yellow', mode='overwrite')

In [22]:
df_join = df_green_revenue.join(df_yellow_revenue, on=['hour', 'zone_id'], how='outer')

In [23]:
df_join.show(10)

[Stage 30:>                                                         (0 + 1) / 1]

+-------------------+-------+------------------+--------------------+------------------+---------------------+
|               hour|zone_id|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+-------+------------------+--------------------+------------------+---------------------+
|2019-01-01 00:00:00|     16|50.599999999999994|                   2|              NULL|                 NULL|
|2019-01-01 00:00:00|     19|              NULL|                NULL|               4.3|                    1|
|2019-01-01 00:00:00|     33| 551.1400000000001|                  25|428.08000000000004|                   19|
|2019-01-01 00:00:00|     41| 1307.729999999998|                  97|1101.6299999999987|                   83|
|2019-01-01 00:00:00|     52|             18.96|                   1|122.93999999999998|                    6|
|2019-01-01 00:00:00|     61| 330.2600000000001|                  17|113.90999999999998|                    5|
|

In [24]:
df_join.coalesce(1).write.parquet('gs://nyc-tlc-data-lake/report/revenue/total', mode='overwrite')

In [25]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('gs://nyc-tlc-data-lake/csv/taxi_zone_lookup.csv')

In [26]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [27]:
df_zones_revenue = df_join.join(df_zones, df_join.zone_id == df_zones.LocationID)

In [28]:
df_zones_revenue = df_zones_revenue.drop('LocationID', 'zone_id')

In [29]:
df_zones_revenue.show(5)

[Stage 43:>                                                         (0 + 1) / 1]

+-------------------+------------------+--------------------+------------------+---------------------+---------+----------------+------------+
|               hour|      green_amount|green_number_records|     yellow_amount|yellow_number_records|  Borough|            Zone|service_zone|
+-------------------+------------------+--------------------+------------------+---------------------+---------+----------------+------------+
|2019-01-01 00:00:00|50.599999999999994|                   2|              NULL|                 NULL|   Queens|         Bayside|   Boro Zone|
|2019-01-01 00:00:00|              NULL|                NULL|               4.3|                    1|   Queens|       Bellerose|   Boro Zone|
|2019-01-01 00:00:00| 551.1400000000001|                  25|428.08000000000004|                   19| Brooklyn|Brooklyn Heights|   Boro Zone|
|2019-01-01 00:00:00| 1307.729999999998|                  97|1101.6299999999987|                   83|Manhattan|  Central Harlem|   Boro Zone|

In [30]:
df_zones_revenue.coalesce(1).write.parquet('gs://nyc-tlc-data-lake/report/revenue/zones', mode='overwrite')